# 29 — Budget-Matched Comparisons

## When does revision still outperform reset?

Notebook 23 asked what residues survive.

Notebook 29 asks:

> Under matched or explicit compute budgets, when does pruning-as-revision remain useful?

Outputs:

```text
results/29_budget_matched_comparisons.csv
figures/29_budget_matched_comparisons.png
```

No model downloads. No GPU. No pruning jobs.

In [ ]:
# Cell 1 — locate or clone repo

from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/thinkthoughts/pruning-rml.git"
REPO_NAME = "pruning-rml"

cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME:
    repo = cwd
elif cwd.name == "notebooks" and cwd.parent.name == REPO_NAME:
    repo = cwd.parent
elif Path("/content/pruning-rml").exists():
    repo = Path("/content/pruning-rml")
elif (cwd / REPO_NAME).exists():
    repo = cwd / REPO_NAME
else:
    target = Path("/content") if Path("/content").exists() else cwd
    os.chdir(target)
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo = target / REPO_NAME

os.chdir(repo)

src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print("repo:", repo)
print("src :", src)
print("src exists:", src.exists())

In [ ]:
# Cell 2 — ensure helpers

pkg = repo / "src" / "pruning_rml"
pkg.mkdir(parents=True, exist_ok=True)

(pkg / "__init__.py").write_text('__version__ = "0.1.0"\n', encoding="utf-8")

(pkg / "paths.py").write_text('''from pathlib import Path

def ensure_dirs(root, names=("results", "figures", "reports")):
    root = Path(root)
    outputs = {}
    for name in names:
        path = root / name
        path.mkdir(parents=True, exist_ok=True)
        outputs[name] = path
    return outputs
''', encoding="utf-8")

(pkg / "budgets.py").write_text('''def budget_ratio(candidate_tokens, baseline_tokens):
    if baseline_tokens <= 0:
        raise ValueError("baseline_tokens must be positive")
    return candidate_tokens / baseline_tokens

def token_efficiency(score, tokens):
    if tokens <= 0:
        raise ValueError("tokens must be positive")
    return score / tokens

def budget_label(ratio):
    if ratio == 1:
        return "matched"
    if ratio < 1:
        return "smaller"
    return "larger"
''', encoding="utf-8")

for module_name in [
    "pruning_rml",
    "pruning_rml.paths",
    "pruning_rml.budgets",
]:
    if module_name in sys.modules:
        del sys.modules[module_name]

print("helpers ready")
print("package files:", sorted(p.name for p in pkg.iterdir()))

In [ ]:
# Cell 3 — imports and output folders

import pandas as pd
import matplotlib.pyplot as plt

from pruning_rml.paths import ensure_dirs
from pruning_rml.budgets import budget_ratio, token_efficiency, budget_label

outputs = ensure_dirs(repo)
results = outputs["results"]
figures = outputs["figures"]

print("results:", results)
print("figures:", figures)

## Budget schema

This notebook uses toy values to encode the comparison logic.

The table is not a reproduction of paper benchmarks.

It separates three ideas:

```text
same budget
    → fair reset-vs-revision comparison

larger scratch budget
    → reset can catch up

fine revision
    → may preserve inherited structure longer
```

In [ ]:
# Cell 4 — build budget comparison table

baseline_tokens = 100

rows = [
    {
        "condition": "scratch matched budget",
        "mode": "reset",
        "tokens": 100,
        "relative_score": 0.72,
        "interpretation": "reset must rebuild capability under equal tokens",
    },
    {
        "condition": "pruned matched budget",
        "mode": "revision",
        "tokens": 100,
        "relative_score": 0.84,
        "interpretation": "revision inherits useful structure under equal tokens",
    },
    {
        "condition": "scratch larger budget",
        "mode": "reset",
        "tokens": 180,
        "relative_score": 0.83,
        "interpretation": "reset can become competitive with more training",
    },
    {
        "condition": "fine pruning continued budget",
        "mode": "revision",
        "tokens": 130,
        "relative_score": 0.87,
        "interpretation": "finer revision may retain advantage longer",
    },
]

df = pd.DataFrame(rows)
df["budget_ratio"] = df["tokens"].apply(lambda x: budget_ratio(x, baseline_tokens))
df["budget_label"] = df["budget_ratio"].apply(budget_label)
df["token_efficiency"] = df.apply(
    lambda row: token_efficiency(row["relative_score"], row["tokens"]),
    axis=1,
)

df

In [ ]:
# Cell 5 — save CSV

csv_path = results / "29_budget_matched_comparisons.csv"
df.to_csv(csv_path, index=False)

print("saved:", csv_path)
print("exists:", csv_path.exists())

## Visual summary

The figure compares relative scores while making the budget condition visible in the labels.

Working hypothesis:

> revision is most useful when inherited structure survives enough to reduce the amount of relearning required.

In [ ]:
# Cell 6 — save figure

fig_path = figures / "29_budget_matched_comparisons.png"

plot_df = df.copy()
plot_df["label"] = plot_df["condition"] + "\n(" + plot_df["tokens"].astype(str) + " tokens)"

ax = plot_df.plot.bar(
    x="label",
    y="relative_score",
    legend=False,
    figsize=(10, 5),
)

ax.set_title("Budget-matched comparisons: reset vs revision")
ax.set_xlabel("Condition")
ax.set_ylabel("Relative score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(fig_path, dpi=160)
plt.show()

print("saved:", fig_path)
print("exists:", fig_path.exists())

In [ ]:
# Cell 7 — verify outputs

expected = [
    results / "29_budget_matched_comparisons.csv",
    figures / "29_budget_matched_comparisons.png",
]

for path in expected:
    print(path, "exists:", path.exists())

## Result

Notebook 29 creates:

```text
results/29_budget_matched_comparisons.csv
figures/29_budget_matched_comparisons.png
```

Next notebook:

```text
31_revision_triggers.ipynb
```

Next question:

> Can a system detect when revision is needed before reset becomes necessary?